# 🔄 eidcloud-offline-sync Quickstart

This notebook demonstrates **zero-dependency offline-first distributed synchronization** using SQLite, Conflict-Free Replicated Data Types (**CRDTs**), and Hybrid Logical Clocks (**HLC**) in pure PHP.

### Workflow Steps Demonstrated:
1. **Environment Setup & Node Initialization**: Provisioning Node Alpha and Node Beta.
2. **Concurrent Offline Mutations**: Independent local writes simulating network partition.
3. **Peer Sync & Conflict-Free Convergence**: LWW resolution with deterministic tie-breaking and conflict auditing.

In [ ]:
# Cell 1: Clone Repository and Verify PHP 8.2+ Environment
!apt-get update -qq && apt-get install -y -qq php-cli php-sqlite3 > /dev/null
!php -v

# Check if running locally or in Colab
import os
if not os.path.exists('src/SyncEngine.php'):
    !git clone https://github.com/shadi-alhasan/eidcloud-offline-sync.git .

!php tests/run_tests.php

In [ ]:
# Cell 2: Simulate Two Offline Nodes Making Concurrent Writes
script = '''<?php
require_once 'src/Clock/Timestamp.php';
require_once 'src/Clock/HybridLogicalClock.php';
require_once 'src/Crdt/LwwRegister.php';
require_once 'src/Storage/SqliteStore.php';
require_once 'src/Queue/MutationQueue.php';
require_once 'src/SyncEngine.php';

use EidCloud\\OfflineSync\\SyncEngine;

// Initialize Node Alpha and Node Beta
$nodeA = SyncEngine::open('alpha.sqlite', 'node-alpha');
$nodeB = SyncEngine::open('beta.sqlite', 'node-beta');

echo "=== Phase 1: Shared Initial Seed ===\n";
$nodeA->setField('inventory', 'item-101', 'name', 'Field Tablet Pro');
$nodeA->setField('inventory', 'item-101', 'stock', '50');

// Initial sync to replicate seed to Node B
$nodeA->syncWithPeer($nodeB);
echo "Synced seed state from Alpha to Beta.\n\n";

echo "=== Phase 2: Partition & Concurrent Offline Edits ===\n";
// Node A edits offline
$tsA = $nodeA->setField('inventory', 'item-101', 'stock', '48');
echo "Node Alpha offline edit: stock = 48 (HLC: {$tsA})\n";

// Node B edits concurrently offline (e.g. slight physical drift/later write)
usleep(3000);
$tsB = $nodeB->setField('inventory', 'item-101', 'stock', '42');
$nodeB->setField('inventory', 'item-101', 'warehouse', 'Sector-7');
echo "Node Beta offline edit:  stock = 42 (HLC: {$tsB})\n";
echo "Node Beta offline edit:  warehouse = Sector-7\n";
'''
with open('simulate_offline.php', 'w') as f:
    f.write(script)

!php simulate_offline.php

In [ ]:
# Cell 3: Peer Synchronization, Conflict Audit & Convergence Verification
sync_script = '''<?php
require_once 'src/Clock/Timestamp.php';
require_once 'src/Clock/HybridLogicalClock.php';
require_once 'src/Crdt/LwwRegister.php';
require_once 'src/Storage/SqliteStore.php';
require_once 'src/Queue/MutationQueue.php';
require_once 'src/SyncEngine.php';

use EidCloud\\OfflineSync\\SyncEngine;

$nodeA = SyncEngine::open('alpha.sqlite', 'node-alpha');
$nodeB = SyncEngine::open('beta.sqlite', 'node-beta');

echo "=== Phase 3: Peer Reconnection & Delta Sync ===\n";
$summary = $nodeA->syncWithPeer($nodeB);

echo "Synchronization Summary:\n";
echo json_encode($summary, JSON_PRETTY_PRINT) . "\n\n";

echo "=== Phase 4: State Convergence Verification ===\n";
$stateA = $nodeA->getStore()->getFieldMetadata('inventory', 'item-101', 'stock');
$stateB = $nodeB->getStore()->getFieldMetadata('inventory', 'item-101', 'stock');

echo "Node Alpha Stock: {$stateA['value']} (HLC: {$stateA['hlc']})\n";
echo "Node Beta  Stock: {$stateB['value']} (HLC: {$stateB['hlc']})\n";

if ($stateA['value'] === $stateB['value']) {
    echo "✅ SUCCESS: Converged to deterministic state: '{$stateA['value']}'\n";
} else {
    echo "❌ FAILURE: States diverged!\n";
}

echo "\n=== Phase 5: Conflict Audit Trail on Node Alpha ===\n";
$conflicts = $nodeA->getStore()->getConflicts();
foreach ($conflicts as $c) {
    echo "Conflict on [{$c['table_name']}.{$c['column_name']}]: Winner HLC={$c['winner_hlc']} (Val: {$c['winner_val']}), Loser HLC={$c['loser_hlc']} (Val: {$c['loser_val']})\n";
}
'''
with open('verify_sync.php', 'w') as f:
    f.write(sync_script)

!php verify_sync.php